<a href="https://colab.research.google.com/github/swrobuts/e-comm/blob/main/notebooks/sentiment_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Sentiment-Analyse mit Transformer (RoBERTa)
### UrbanCart reviews.csv · THWS E-Commerce Kurs · Bonus-Lab

Dieses Notebook nutzt **cardiffnlp/twitter-roberta-base-sentiment-latest** – ein auf Social-Media-Texten vortrainiertes RoBERTa-Modell – sowie als spezialisierte Alternative **`pysentimiento`** für E-Commerce-nahe Texte.

**Was ist anders als VADER?**

| | VADER | Transformer |
|---|---|---|
| Versteht Kontext | ❌ | ✅ |
| Versteht Ironie | ❌ | Teilweise ✅ |
| Setup | Sofort | ~2 GB Download |
| Geschwindigkeit | Sehr schnell | Langsamer (GPU empfohlen) |
| Accuracy (E-Commerce) | ~65 % | ~80–85 % |

**Was wir machen:**
1. RoBERTa-Modell laden (via Hugging Face)
2. Batch-Inferenz auf reviews.csv
3. Vergleich VADER vs. Transformer
4. Aspekt-basierte Analyse mit `pysentimiento` (optional)
5. Fehleranalyse: Wo liegen die Modelle falsch?

---
> ⚡ **GPU empfohlen:** In Colab unter `Laufzeit → Laufzeittyp ändern → T4 GPU` aktivieren.

In [ ]:
# ── Schritt 0: Pakete installieren ──────────────────────────────────────────
# Hinweis: transformers + torch = ~2 GB Download, dauert in Colab ~2–3 Minuten
!pip install transformers torch pandas plotly pysentimiento --quiet
print("✅ Pakete installiert")

In [ ]:
# ── Schritt 1: Daten laden ──────────────────────────────────────────────────
import pandas as pd

URL = "https://raw.githubusercontent.com/swrobuts/e-comm/main/data/reviews.csv"
df = pd.read_csv(URL)

print(f"✅ Geladen: {len(df):,} Reviews")
df.head(3)

In [ ]:
# ── Schritt 2: RoBERTa-Modell laden ─────────────────────────────────────────
from transformers import pipeline

# Cardiff NLP RoBERTa – auf 58 Mio Tweets trainiert
# Labels: negative / neutral / positive
MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

print(f"Lade Modell: {MODEL}")
print("(~500 MB Download, einmalig – danach gecacht)")

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=MODEL,
    tokenizer=MODEL,
    max_length=512,
    truncation=True,
    device=0  # GPU wenn verfügbar; -1 für CPU
)
print("✅ Modell geladen")

In [ ]:
# ── Schritt 3: Schnelltest ───────────────────────────────────────────────────
test_texts = [
    "Absolutely love this product! Fast shipping, great quality.",
    "It's okay, nothing special. Would not buy again.",
    "Terrible quality. Broke after 2 days. Complete waste of money!",
    "Not bad for the price, but the packaging was damaged on arrival."
]

print("Schnelltest:")
for text in test_texts:
    result = sentiment_pipe(text)[0]
    label = result['label'].lower()
    score = result['score']
    emoji = {'positive': '🟢', 'neutral': '🟡', 'negative': '🔴'}.get(label, '⚪')
    print(f"  {emoji} {label:8s} ({score:.2f}) | {text[:60]}...")

In [ ]:
# ── Schritt 4: Batch-Inferenz auf allen Reviews ──────────────────────────────
# Hinweis: auf CPU ~15 Min für 5.800 Reviews; auf T4-GPU ~1–2 Min
from tqdm.auto import tqdm

texts = df['review_text'].astype(str).tolist()

print(f"Analysiere {len(texts):,} Reviews...")
results = []
batch_size = 64

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_results = sentiment_pipe(batch)
    results.extend(batch_results)

df['roberta_label'] = [r['label'].lower() for r in results]
df['roberta_score'] = [r['score'] for r in results]

print("\n✅ Fertig!")
print("RoBERTa-Label-Verteilung:")
print(df['roberta_label'].value_counts())

In [ ]:
# ── Schritt 5: Accuracy vs. Original-Labels ──────────────────────────────────

# Map original German sentiment labels to English for consistent comparison
sentiment_mapping = {
    'positiv': 'positive',
    'neutral': 'neutral',
    'negativ': 'negative'
}
df['sentiment_mapped'] = df['sentiment'].map(sentiment_mapping)

# RoBERTa Accuracy
roberta_acc = (df['sentiment_mapped'] == df['roberta_label']).mean() * 100
print(f"RoBERTa Accuracy: {roberta_acc:.1f}%")

# Vergleich mit VADER
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
df['vader_compound'] = df['review_text'].astype(str).apply(
    lambda t: analyzer.polarity_scores(t)['compound']
)
df['vader_label'] = df['vader_compound'].apply(
    lambda s: 'positive' if s >= 0.05 else ('negative' if s <= -0.05 else 'neutral')
)
vader_acc = (df['sentiment_mapped'] == df['vader_label']).mean() * 100
print(f"VADER Accuracy:   {vader_acc:.1f}%")
print(f"→ RoBERTa ist um {roberta_acc - vader_acc:.1f}%-Punkte besser")

print("\nKonfusionsmatrix RoBERTa vs. Original:")
# Use df['sentiment_mapped'] for the confusion matrix as well
print(pd.crosstab(df['sentiment_mapped'], df['roberta_label'], margins=True))

In [ ]:
# ── Schritt 6: Visualisierung – Modellvergleich ──────────────────────────────
import plotly.graph_objects as go

labels = ['positive', 'neutral', 'negative']
colors = {'positive': '#E87722', 'neutral': '#888888', 'negative': '#005B9A'}

fig = go.Figure()

for model_col, model_name in [('sentiment', 'Original'), ('vader_label', 'VADER'), ('roberta_label', 'RoBERTa')]:
    counts = df[model_col].value_counts()
    fig.add_trace(go.Bar(
        name=model_name,
        x=labels,
        y=[counts.get(l, 0) for l in labels],
    ))

fig.update_layout(
    barmode='group',
    title='Sentiment-Verteilung: Original vs. VADER vs. RoBERTa',
    xaxis_title='Sentiment',
    yaxis_title='Anzahl Reviews',
    template='simple_white',
    legend_title='Modell'
)
fig.show()

In [ ]:
# ── Schritt 7: Fehleranalyse – wo irrt sich RoBERTa? ────────────────────────
errors = df[df['sentiment'] != df['roberta_label']].copy()

print(f"Falsch klassifiziert: {len(errors):,} von {len(df):,} ({len(errors)/len(df)*100:.1f}%)")
print("\nTypische Fehlerfälle:")

# Zeige je 2 Beispiele pro Fehlertyp
for true_label in ['positive', 'neutral', 'negative']:
    for pred_label in ['positive', 'neutral', 'negative']:
        if true_label == pred_label:
            continue
        subset = errors[(errors['sentiment'] == true_label) & (errors['roberta_label'] == pred_label)]
        if len(subset) > 0:
            example = subset.sample(1, random_state=42).iloc[0]
            print(f"\n  Wahr: {true_label:8s} → Vorhergesagt: {pred_label:8s}")
            print(f"  Score: {example['roberta_score']:.2f} | {str(example['review_text'])[:100]}...")

In [ ]:
# ── Schritt 8: pysentimiento – spezialisiert auf Social-Media/E-Commerce ─────
# pysentimiento: vortrainiert auf Twitter, Review-Daten; multi-class, multi-lingual
from pysentimiento import create_analyzer

print("Lade pysentimiento Analyzer...")
analyzer_ps = create_analyzer(task="sentiment", lang="en")
print("✅ Geladen")

# Test auf 5 Beispielen
print("\nTest pysentimiento:")
sample = df.sample(5, random_state=7)
for _, row in sample.iterrows():
    result = analyzer_ps.predict(str(row['review_text'])[:512])
    emoji = {'POS': '🟢', 'NEU': '🟡', 'NEG': '🔴'}.get(result.output, '⚪')
    print(f"  {emoji} pysentimiento={result.output:3s} | original={row['sentiment']:8s} | {str(row['review_text'])[:60]}...")

In [ ]:
# ── Schritt 9: Vollständige Batch-Analyse mit pysentimiento ──────────────────
# (Optional – dauert länger als RoBERTa)
# Kommentar entfernen um auszuführen:

# ps_labels_map = {'POS': 'positive', 'NEU': 'neutral', 'NEG': 'negative'}
# print("Analysiere alle Reviews mit pysentimiento...")
# ps_results = []
# for text in tqdm(df['review_text'].astype(str).tolist()):
#     try:
#         r = analyzer_ps.predict(text[:512])
#         ps_results.append(ps_labels_map.get(r.output, 'neutral'))
#     except:
#         ps_results.append('neutral')
# df['pysentimiento_label'] = ps_results
# ps_acc = (df['sentiment'] == df['pysentimiento_label']).mean() * 100
# print(f"pysentimiento Accuracy: {ps_acc:.1f}%")

print("(Batch-Analyse auskommentiert – entferne # zum Aktivieren)")

## 🎯 Zusammenfassung & Modellvergleich

| Merkmal | VADER | RoBERTa (Cardiff NLP) | pysentimiento |
|---|---|---|---|
| **Typ** | Regelbasiert | Transformer | Transformer |
| **Modellgröße** | ~0 MB | ~500 MB | ~500 MB |
| **Setup** | Sofort | ~2 Min Download | ~2 Min Download |
| **Accuracy (E-Commerce)** | ~65 % | ~80–85 % | ~78–82 % |
| **Versteht Ironie** | ❌ | Teilweise | Teilweise |
| **Mehrsprachig** | Englisch | Englisch | Englisch + Spanisch + Portugiesisch |
| **GPU nötig?** | Nein | Empfohlen | Empfohlen |
| **Ideal für** | Schnelle Übersicht | Höchste Genauigkeit | Social-Media + Reviews |

### Empfehlung für UrbanCart
- **Explorative Analyse / Prototyp:** VADER → schnell, kein Setup
- **Produktionssystem:** RoBERTa → bessere Accuracy, skalierbar via API
- **Aspekt-basiert (Lieferung, Qualität, Preis):** PyABSA oder spezialisierte ABSA-Modelle

---
*THWS E-Commerce Kurs · Bonus-Lab · Prof. R. Butscher*